In [1]:
from decouple import config
from sqlalchemy import create_engine
from sqlalchemy import text
import pandas as pd
import oracledb
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim
import os
from datetime import datetime
from pyspark.sql.functions import to_date

In [2]:
os.environ['SPARK_HOME'] = '/opt/spark/'
# Configuración de Spark
spark = SparkSession.builder \
    .appName("etl") \
    .master("local[*]") \
    .config("spark.executor.memory", "24g") \
    .config("spark.executor.cores", "8") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .getOrCreate()

24/02/23 10:06:14 WARN Utils: Your hostname, linux01 resolves to a loopback address: 127.0.0.1; using 10.0.28.15 instead (on interface eno1)
24/02/23 10:06:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/02/23 10:06:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
tabla='ctaam10'
col_tabla='atenambatenfec'
col_dat='ate_fec'
dat='DAT_CEXT002_ESSI'

In [4]:
# Oracle Conexion
DB_USER_ORACLE = config("USER4_BDI_POSTGRES")
DB_PASSWORD_ORACLE = config("PASS4_BDI_POSTGRES")
DB_NAME_ORACLE = "WNET"
DB_PORT_ORACLE = "1527"
DB_HOST_ORACLE = config("HOST4_BDI_POSTGRES")
cadena0 = f"jdbc:oracle:thin:@//{DB_HOST_ORACLE}:{DB_PORT_ORACLE}/{DB_NAME_ORACLE}"

DB_USER_ORACLE_DW = 'DW_ESSALUD'
DB_PASSWORD_ORACLE_DW = 'Dw_Essalud24'
DB_NAME_ORACLE_DW = "devugad"
DB_PORT_ORACLE_DW = "1521"
DB_HOST_ORACLE_DW = '10.0.1.228'
cadena4 = f"jdbc:oracle:thin:@//{DB_HOST_ORACLE_DW}:{DB_PORT_ORACLE_DW}/{DB_NAME_ORACLE_DW}"

# PostgreSQL Conexiones
DB_USER = config("USER2_BDI_POSTGRES")
DB_PASSWORD = config("PASS2_BDI_POSTGRES")
DB_PORT = "5432"
DB_HOST = config("HOST2_BDI_POSTGRES")

# Connection for essi_etl
DB_NAME_ESSI_ETL = "essi_etl"
url1 = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME_ESSI_ETL}"
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME_ESSI_ETL}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

# Connection for dw_essalud
DB_NAME_DW_ESSALUD = "dw_essalud"
url2 = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME_DW_ESSALUD}"
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME_DW_ESSALUD}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

# Connection for dl_essi
DB_NAME_DL_ESSI = "dl_essi"
url3 = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME_DL_ESSI}"
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME_DL_ESSI}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [5]:
query = "SELECT fec_ini FROM etl_act where id_mod=4"

fecha = spark.read \
    .format("jdbc") \
    .option("url", cadena4)\
    .option("query", query) \
    .option("user", DB_USER_ORACLE_DW) \
    .option("password", DB_PASSWORD_ORACLE_DW) \
    .option("driver", "oracle.jdbc.OracleDriver") \
    .load()

fecha = fecha.collect()[0][0]
# Convertir la cadena a un objeto datetime
fecha_sin_hora = fecha.date()
print(fecha_sin_hora)

2019-01-01


In [6]:
fecha='2024-02-01'

In [7]:
query = f"""
	SELECT
		to_char(d1.ATENAMBATENFEC, 'YYYY-MM-DD HH24:MI:SS') as ATENAMBATENFEC,
		d1.ATENAMBTIPCONCOD,
		d1.ATENAMBCSECOD,
		d1.CPSCOD,
		d1.ATENAMBNUMATECOD,
		d1.TIPCONTLEYCOD,
		d1.RESATENAMBUCOD,
		d1.ORICENASIREFCOD,
		d1.CENASIREFCOD,
		d1.ATENAMBESTREGCOD,
		d1.ATENAMBUSUCRECOD,
		to_char(d1.ATENAMBCREFEC, 'YYYY-MM-DD HH24:MI:SS') as ATENAMBCREFEC,
		d1.ATENAMBUSUMODCOD,
		to_char(d1.ATENAMBMODFEC, 'YYYY-MM-DD HH24:MI:SS') as ATENAMBMODFEC,
		d1.ATENAMBULTREGFEC,
		d1.ATENAMBPACSECNUM,
		per.PERTIPDOCIDENCOD,
		per.PERDOCIDENNUM,
		per.PERNACFEC,
		per.PERSEXOCOD,
		a1.*
	FROM ctaam10 d1 
	LEFT OUTER JOIN CTDAA10 a1 ON a1.ATENAMBORICENASICOD = d1.ATENAMBORICENASICOD
		AND a1.ATENAMBCENASICOD    = d1.ATENAMBCENASICOD
		AND a1.ATENAMBNUM    = d1.ATENAMBNUM
	LEFT OUTER JOIN CMPER10 per ON d1.ATENAMBPACSECNUM = per.PERSECNUM
	WHERE d1.{col_tabla} >= TO_DATE('2024-02-22', 'YYYY-MM-DD')
	"""

base = spark.read \
    .format("jdbc") \
    .option("url", cadena0)\
    .option("query", query) \
    .option("user", DB_USER_ORACLE) \
    .option("password", DB_PASSWORD_ORACLE) \
    .option("driver", "oracle.jdbc.OracleDriver") \
    .load()

In [8]:
# Lista de columnas a eliminar
columnas_eliminar = ['ATENAMBOBS','ATENAMBANAM','ATENAMBEXAMCLIN','ATENAMBPACIND','ATENAMBPLANTRA','ATENAMBSOLOPE','oricenasirefcod', 'cenasirefcod', 'diagatenambpeas']

# Eliminar las columnas
base = base.drop(*columnas_eliminar)

# Renombrar columnas
base = base.withColumnRenamed('atenamboricenasicod', 'ori_cas') \
    .withColumnRenamed('atenambcenasicod', 'cod_cas') \
    .withColumnRenamed('atenambnum', 'ate_num') \
    .withColumnRenamed('atenambatenfec', 'ate_fec') \
    .withColumnRenamed('atenambtipconcod', 'tip_con') \
    .withColumnRenamed('atenambcsecod', 'cod_cse') \
    .withColumnRenamed('cpscod', 'cod_cps') \
    .withColumnRenamed('atenambnumatecod', 'ate_cod') \
    .withColumnRenamed('tipcontleycod', 'ley_cod') \
    .withColumnRenamed('resatenambucod', 'res_cod') \
    .withColumnRenamed('atenambestregcod', 'est_reg') \
    .withColumnRenamed('atenambusucrecod', 'usu_cre') \
    .withColumnRenamed('atenambcrefec', 'fec_cre') \
    .withColumnRenamed('atenambusumodcod', 'usu_mod') \
    .withColumnRenamed('atenambmodfec', 'fec_mod') \
    .withColumnRenamed('atenambultregfec', 'fec_ult_reg') \
    .withColumnRenamed('atenambpacsecnum', 'pac_sec') \
    .withColumnRenamed('pertipdocidencod', 'cod_tdi_pac') \
    .withColumnRenamed('perdocidennum', 'num_doc_pac') \
    .withColumnRenamed('conddiagcod', 'cond_diagcod') \
    .withColumnRenamed('diagcod', 'diag_cod') \
    .withColumnRenamed('atenambdiagord', 'diag_ord') \
    .withColumnRenamed('atenambtipodiagcod', 'tip_diag') \
    .withColumnRenamed('atenambcasodiagcod', 'caso_diag') \
    .withColumnRenamed('diagatenambaltaflag', 'alta_diag') \
    .withColumnRenamed('pernacfec', 'fec_nac_pac') \
    .withColumnRenamed('persexocod', 'sexo_pac')


In [9]:
query = f"SELECT * FROM {dat} WHERE ROWNUM <= 10"
base1 = spark.read \
    .format("jdbc") \
    .option("url", cadena4)\
    .option("query", query) \
    .option("user", DB_USER_ORACLE_DW) \
    .option("password", DB_PASSWORD_ORACLE_DW) \
    .option("driver", "oracle.jdbc.OracleDriver") \
    .load()

In [10]:
query = f"SELECT id_oricas,ori_cod FROM dim_oricas"
ori = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
ori = ori.withColumnRenamed('ori_cod', 'ori_cas')
base = base.join(ori, how="left", on="ori_cas")
base = base.drop('ori_cas')

In [11]:
query = f"SELECT id_cas, id_red, cod_cas FROM dim_cas where id_red is not null"
cas = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
cas = cas.dropna()
base = base.join(cas, how="left", on="cod_cas")
base = base.drop('cod_cas')

In [12]:
query = f"SELECT id_tipcon, cod_tipcon FROM dim_tipcon"
tipcon = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
tipcon = tipcon.withColumnRenamed('cod_tipcon', 'tip_con')
tipcon = tipcon.withColumn('tip_con', trim(col('tip_con')))
base = base.withColumn('tip_con', trim(col('tip_con')))
base = base.join(tipcon, how="left", on="tip_con")
base = base.drop('tip_con')

In [13]:
query = f"SELECT id_csep, cod_cse FROM dim_csep"
csep = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
csep = csep.withColumn('cod_cse', trim(col('cod_cse')))
base = base.withColumn('cod_cse', trim(col('cod_cse')))
base = base.join(csep, how="left", on="cod_cse")
base = base.drop('cod_cse')

In [14]:
query = f"SELECT id_cps,cod_cps FROM dim_cps"
cps = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
cps = cps.withColumn('cod_cps', trim(col('cod_cps')))
base = base.withColumn('cod_cps', trim(col('cod_cps')))
base = base.join(cps, how="left", on="cod_cps")
base = base.drop('cod_cps')

In [15]:
query = f"SELECT id_numaten, cod_numaten FROM dim_numaten"
numaten = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
numaten = numaten.withColumnRenamed('cod_numaten', 'ate_cod')
numaten = numaten.withColumn('ate_cod', trim(col('ate_cod')))
base = base.withColumn('ate_cod', trim(col('ate_cod')))
base = base.join(numaten, how="left", on="ate_cod")
base = base.drop('ate_cod')

In [16]:
query = f"SELECT id_tipleycont, cod_tipleycont FROM dim_tipleycont"
ley = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
ley = ley.withColumnRenamed('cod_tipleycont', 'ley_cod')
ley = ley.withColumn('ley_cod', trim(col('ley_cod')))
base = base.withColumn('ley_cod', trim(col('ley_cod')))
base = base.join(ley, how="left", on="ley_cod")
base = base.drop('ley_cod')

In [17]:
query = f"SELECT id_resaten, cod_resaten FROM dim_resaten"
resaten = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
resaten = resaten.withColumnRenamed('cod_resaten', 'res_cod')
resaten = resaten.withColumn('res_cod', trim(col('res_cod')))
base = base.withColumn('res_cod', trim(col('res_cod')))
base = base.join(resaten, how="left", on="res_cod")
base = base.drop('res_cod')

In [18]:
query = f"SELECT id_estreg, cod_est FROM dim_estreg"
estreg = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
estreg = estreg.withColumnRenamed('cod_est', 'est_reg')
estreg = estreg.withColumn('est_reg', trim(col('est_reg')))
base = base.withColumn('est_reg', trim(col('est_reg')))
base = base.join(estreg, how="left", on="est_reg")
base = base.drop('est_reg')

In [19]:
query = f"SELECT id_condia,cod_con FROM dim_condiag"
condiag = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
base = base.withColumnRenamed('cond_diagcod', 'cod_con')
condiag = condiag.withColumn('cod_con', trim(col('cod_con')))
base = base.withColumn('cod_con', trim(col('cod_con')))
base = base.join(condiag, how="left", on="cod_con")
base = base.drop('cod_con')

In [20]:
query = f"SELECT id_cie,cod_cie FROM dim_cie10"
cie10 = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
base = base.withColumnRenamed('diag_cod', 'cod_cie')
cie10 = cie10.withColumn('cod_cie', trim(col('cod_cie')))
base = base.withColumn('cod_cie', trim(col('cod_cie')))
base = base.join(cie10, how="left", on="cod_cie")
base = base.drop('cod_cie')

In [21]:
query = f"SELECT id_tipdx,cod_tdx FROM dim_tipdx"
tipdx = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
tipdx = tipdx.withColumnRenamed('cod_tdx', 'tip_diag')
tipdx = tipdx.withColumn('tip_diag', trim(col('tip_diag')))
base = base.withColumn('tip_diag', trim(col('tip_diag')))
base = base.join(tipdx, how="left", on="tip_diag")
base = base.drop('tip_diag')

In [22]:
query = f"SELECT id_casdiag,cod_casdiag FROM dim_casdiag"
casdiag = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
casdiag = casdiag.withColumnRenamed('cod_casdiag', 'caso_diag')
casdiag = casdiag.withColumn('caso_diag', trim(col('caso_diag')))
base = base.withColumn('caso_diag', trim(col('caso_diag')))
base = base.join(casdiag, how="left", on="caso_diag")
base = base.drop('caso_diag')

In [23]:
query = f"SELECT id_tipdoc,cod_tdo FROM dim_tipdoc"
tipdoc = spark.read \
    .format("jdbc") \
    .option("url", url2)\
    .option("query", query) \
    .option("user", DB_USER) \
    .option("password", DB_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .load()
tipdoc = tipdoc.withColumnRenamed('id_tipdoc', 'id_tdi_pac')
tipdoc = tipdoc.withColumnRenamed('cod_tdo', 'cod_tdi_pac')
tipdoc = tipdoc.withColumn('cod_tdi_pac', trim(col('cod_tdi_pac')))
base = base.withColumn('cod_tdi_pac', trim(col('cod_tdi_pac')))
base = base.join(tipdoc, how="left", on="cod_tdi_pac")
base = base.drop('cod_tdi_pac')

In [24]:
base.schema

StructType([StructField('ate_fec', StringType(), True), StructField('usu_cre', StringType(), True), StructField('fec_cre', StringType(), True), StructField('usu_mod', StringType(), True), StructField('fec_mod', StringType(), True), StructField('fec_ult_reg', TimestampType(), True), StructField('pac_sec', DecimalType(10,0), True), StructField('num_doc_pac', StringType(), True), StructField('fec_nac_pac', TimestampType(), True), StructField('sexo_pac', StringType(), True), StructField('ate_num', DecimalType(10,0), True), StructField('diag_ord', DecimalType(2,0), True), StructField('alta_diag', StringType(), True), StructField('id_oricas', IntegerType(), True), StructField('id_cas', LongType(), True), StructField('id_red', DecimalType(2,0), True), StructField('id_tipcon', IntegerType(), True), StructField('id_csep', LongType(), True), StructField('id_cps', IntegerType(), True), StructField('id_numaten', LongType(), True), StructField('id_tipleycont', LongType(), True), StructField('id_res

In [25]:
base = base.withColumn('ATE_FEC', to_date(base['ATE_FEC'], 'yyyy-MM-dd'))
base = base.withColumn('FEC_CRE', to_date(base['FEC_CRE'], 'yyyy-MM-dd'))
base = base.withColumn('FEC_MOD', to_date(base['FEC_MOD'], 'yyyy-MM-dd'))
base = base.withColumn('FEC_ULT_REG', to_date(base['FEC_ULT_REG'], 'yyyy-MM-dd'))
base = base.withColumn('FEC_NAC_PAC', to_date(base['FEC_NAC_PAC'], 'yyyy-MM-dd'))

In [27]:
numero_lotes = 16
proporciones = [1.0 / numero_lotes] * numero_lotes

# Dividir el DataFrame en lotes aleatorios
lotes = base.randomSplit(proporciones, seed=42)

# Iterar sobre los lotes y escribir en la base de datos
for i, lote in enumerate(lotes):
    lote.cache()
    lote.write \
        .format("jdbc") \
        .option("url", cadena4) \
        .option("dbtable", f"{dat}") \
        .option("user", DB_USER_ORACLE_DW) \
        .option("password", DB_PASSWORD_ORACLE_DW) \
        .option("driver", "oracle.jdbc.OracleDriver") \
        .mode("append") \
        .save()

24/02/23 10:24:13 ERROR Executor: Exception in task 0.0 in stage 268.0 (TID 64)]
org.apache.spark.SparkUpgradeException: [INCONSISTENT_BEHAVIOR_CROSS_VERSION.PARSE_DATETIME_BY_NEW_PARSER] You may get a different result due to the upgrading to Spark >= 3.0:
Fail to parse '2024-02-23 09:19:58' in the new parser. You can set "spark.sql.legacy.timeParserPolicy" to "LEGACY" to restore the behavior before Spark 3.0, or set to "CORRECTED" and treat it as an invalid datetime string.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failToParseDateTimeInNewParserError(QueryExecutionErrors.scala:1368)
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$1.applyOrElse(DateTimeFormatterHelper.scala:149)
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$1.applyOrElse(DateTimeFormatterHelper.scala:142)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.sql.catalyst.util.Is

Py4JJavaError: An error occurred while calling o448.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 44.0 failed 1 times, most recent failure: Lost task 1.0 in stage 44.0 (TID 170) (10.0.28.15 executor driver): org.apache.spark.SparkUpgradeException: [INCONSISTENT_BEHAVIOR_CROSS_VERSION.PARSE_DATETIME_BY_NEW_PARSER] You may get a different result due to the upgrading to Spark >= 3.0:
Fail to parse '2024-02-23 07:23:59' in the new parser. You can set "spark.sql.legacy.timeParserPolicy" to "LEGACY" to restore the behavior before Spark 3.0, or set to "CORRECTED" and treat it as an invalid datetime string.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failToParseDateTimeInNewParserError(QueryExecutionErrors.scala:1368)
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$1.applyOrElse(DateTimeFormatterHelper.scala:149)
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$